# WA Health establishments: a 49-site feature table

## What is this data source?

This notebook uses the Western Australian Department of Health **Health Hospitals (HEALTH-001)** dataset published through Data WA. The official source contains the locations of public and private **Acute Hospitals** and **Nursing Posts** in Western Australia and is updated annually.

## Which API is used?

This notebook uses the public **ArcGIS REST Feature Layer API** provided through the Western Australian Land Information Platform (SLIP). The layer endpoint is:

```text
https://public-services.slip.wa.gov.au/public/rest/services/
SLIP_Public_Services/Health_WFS/MapServer/1
```

The notebook first requests the layer metadata with `?f=pjson` and confirms that the endpoint type is `Feature Layer`. It then calls the layer's `/query` endpoint once with `where=1=1`, `outFields=*`, `returnGeometry=true`, `outSR=4326`, and `f=geojson` to download all statewide records and point geometry. The API is public and the notebook does not use an API key. A dated local GeoJSON snapshot is reused after the first download so repeated model runs do not silently change when the live dataset is updated.

The source gives us an establishment ID and name, point coordinates, whether the establishment is open, its official category, its Public/Private type, and a bed-count field. It does **not** cover every GP, clinic, pharmacy, emergency service, opening hour, service quality, travel route, or a person's ability to obtain care. For that reason, the extracted variables are named `health_establishment` features rather than broad “healthcare accessibility” features.

## What does this notebook produce?

The notebook reads `site_id`, latitude, and longitude from the authoritative `Model_Data` sheet in `model_dataset.xlsx`. It finds the nearest open establishment for every model site and writes a separate one-row-per-`site_id` feature workbook. Targets and unrelated model columns are not copied into that workbook.

| Final output field | Plain meaning | Intended use |
|---|---|---|
| `health_nearest_establishment_distance_m` | Straight-line geodesic distance, in metres, from the model site to the nearest open Acute Hospital or Nursing Post | The only first-round candidate model feature |
| `health_nearest_establishment_category` | Official category of the nearest establishment: `Acute Hospital` or `Nursing Post` | Interpretation and diagnostic checks; not automatically included in the model |
| `health_nearest_establishment_type` | Official type of the nearest establishment: `Public` or `Private` | Interpretation and diagnostic checks; it does not guarantee access |
| `health_nearest_establishment_id` | Official identifier of the nearest establishment | Audit and reproducibility only; not a model feature |
| `health_nearest_establishment_name` | Official name of the nearest establishment | Human review and spot checks only; not a model feature |
| `health_data_status` | Whether extraction was `matched`, `no_open_record`, `invalid_coordinate`, or `source_failed` | Data-quality control only; not a behavioural model feature |

The bed-count field is not extracted because its zeros and values have not been established as comparable measures of current capacity across facility categories. Radius counts are also excluded because the statewide source is sparse and an arbitrary radius would create many zeros. The output is designed for a later `validate="one_to_one"` merge with other teams' feature tables.


## 1. Configuration and reproducibility

The dated GeoJSON snapshot is an immutable external input. If it already exists, the notebook runs without contacting the service. If it is absent, the code first confirms that the REST endpoint is a `Feature Layer`, then downloads all statewide records once. Change `SNAPSHOT_DATE` only when intentionally creating and reviewing a new source version.

In [1]:
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import Request, urlopen
import json

import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from pyproj import Geod


def find_project_root() -> Path:
    # Find the project repository root on the host or in the container.
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / "requirements.txt").is_file() and (path / "data").is_dir():
            return path
    raise FileNotFoundError(
        f"Project root not found from {current}. Expected requirements.txt and data/."
    )


PROJECT_ROOT = find_project_root()
MODEL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "model_dataset.xlsx"

SNAPSHOT_DATE = "2026-08-24"
SOURCE_DIR = (
    PROJECT_ROOT
    / "data"
    / "external_raw"
    / "wa_health_establishments"
    / SNAPSHOT_DATE
)
SOURCE_PATH = SOURCE_DIR / "health_hospitals_health_001.geojson"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "02_imagery_features"
    / "wa-health-establishments"
    / "haoqi-49-location-pilot"
)
FEATURE_PATH = OUTPUT_DIR / "haoqi-04-wa-health-establishment-features.xlsx"

CATALOG_URL = "https://catalogue.data.wa.gov.au/dataset/health-establishments"
LAYER_URL = (
    "https://public-services.slip.wa.gov.au/public/rest/services/"
    "SLIP_Public_Services/Health_WFS/MapServer/1"
)
QUERY_URL = f"{LAYER_URL}/query"
USER_AGENT = "CITS5553-capstone-health-feature-extraction/1.0"

if not MODEL_DATASET_PATH.is_file():
    raise FileNotFoundError(f"Required model input not found: {MODEL_DATASET_PATH}")

SOURCE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root  :", PROJECT_ROOT)
print("Model input   :", MODEL_DATASET_PATH.relative_to(PROJECT_ROOT))
print("Source snapshot:", SOURCE_PATH.relative_to(PROJECT_ROOT))
print("Feature output:", FEATURE_PATH.relative_to(PROJECT_ROOT))

Project root  : /workspace
Model input   : data/processed/model_dataset.xlsx
Source snapshot: data/external_raw/wa_health_establishments/2026-08-24/health_hospitals_health_001.geojson
Feature output: outputs/02_imagery_features/wa-health-establishments/haoqi-49-location-pilot/haoqi-04-wa-health-establishment-features.xlsx


## 2. Read and validate the current model sites

`model_dataset.xlsx` is the project's authoritative modelling foundation. Feature extraction uses only `site_id`, WGS84 coordinates, and CRS. It does not filter on the targets. The present workbook has 49 unique sites and resolves the earlier missing Northampton record; therefore the older 48-row CSV precondition in `04_wa_health.md` is no longer applicable.

In [2]:
model_data = pd.read_excel(
    MODEL_DATASET_PATH,
    sheet_name="Model_Data",
    dtype={"site_id": "string", "coordinate_crs": "string"},
)

site_columns = ["site_id", "latitude", "longitude", "coordinate_crs"]
missing_columns = set(site_columns) - set(model_data.columns)
if missing_columns:
    raise ValueError(f"Model_Data is missing required columns: {sorted(missing_columns)}")

sites = model_data[site_columns].copy()
sites["site_id"] = sites["site_id"].str.strip()
sites["coordinate_crs"] = sites["coordinate_crs"].str.strip().str.upper()
sites["latitude"] = pd.to_numeric(sites["latitude"], errors="coerce")
sites["longitude"] = pd.to_numeric(sites["longitude"], errors="coerce")

if sites["site_id"].isna().any() or sites["site_id"].eq("").any():
    raise ValueError("site_id contains missing or blank values.")
if sites["site_id"].duplicated().any():
    duplicates = sites.loc[sites["site_id"].duplicated(), "site_id"].tolist()
    raise ValueError(f"site_id must be unique. Duplicates: {duplicates}")

sites["valid_coordinate"] = (
    sites["latitude"].between(-90, 90)
    & sites["longitude"].between(-180, 180)
    & sites["coordinate_crs"].eq("EPSG:4326")
)

print(f"Model sites: {len(sites):,}")
print(f"Unique site_id values: {sites['site_id'].nunique():,}")
print(f"Valid EPSG:4326 coordinates: {sites['valid_coordinate'].sum():,}")
print("qa_status distribution:")
print(model_data["qa_status"].value_counts(dropna=False).to_string())
print(sites.head().to_string(index=False))

Model sites: 49
Unique site_id values: 49
Valid EPSG:4326 coordinates: 49
qa_status distribution:
qa_status
ready    49
  site_id   latitude  longitude coordinate_crs valid_coordinate
250303-01 -32.866282 115.705163      EPSG:4326             True
250303-02 -33.359642 115.678041      EPSG:4326             True
250303-03 -33.360624 116.154738      EPSG:4326             True
250303-04 -34.238746 116.149491      EPSG:4326             True
250303-05 -32.628918 115.874612      EPSG:4326             True


## 3. Official dataset scope and metadata

The official [Data WA catalogue record](https://catalogue.data.wa.gov.au/dataset/health-establishments) identifies **Health Hospitals (HEALTH-001)** as a Department of Health dataset covering public and private acute hospitals and nursing posts in Western Australia. It says the list is updated annually. The catalogue currently reports data last updated on 8 July 2025, metadata last updated on 18 August 2026, source projection GDA2020, and “Licence not supplied”; the copyright notice is © Government of Western Australia 2025, Department of Health. The source therefore must not be relabelled as CC BY.

The verified [official Feature Layer](https://public-services.slip.wa.gov.au/public/rest/services/SLIP_Public_Services/Health_WFS/MapServer/1) exposes the fields used here: `estab_id` (Establishment ID), `establishm` (Establishment Name), `is_open` (Open), `category` (Category), `est` (Establishment Type), `no_of_beds` (Number Of Beds), `longitude`, and `latitude`. This source does **not** represent all GPs, clinics, pharmacies, emergency capability, opening hours, service capacity, travel time, or individual access.

## 4. Load one dated statewide snapshot

The live-service check is performed only when a snapshot must be created. The query requests all fields and point geometry in EPSG:4326 in a single statewide call. A cached snapshot is preferred for repeatability and to avoid silently changing feature values between model runs.

In [3]:
def read_json_url(url: str, params: dict | None = None, timeout: int = 120) -> dict:
    # Read JSON from an official endpoint with an explicit user agent.
    full_url = f"{url}?{urlencode(params)}" if params else url
    request = Request(full_url, headers={"User-Agent": USER_AGENT})
    with urlopen(request, timeout=timeout) as response:
        return json.loads(response.read().decode("utf-8"))


source_mode = "cached dated snapshot"
source_error = None

if not SOURCE_PATH.is_file():
    source_mode = "downloaded official snapshot"
    try:
        layer_metadata = read_json_url(LAYER_URL, {"f": "pjson"})
        if layer_metadata.get("type") != "Feature Layer":
            raise RuntimeError(
                f"Expected a Feature Layer, received: {layer_metadata.get('type')!r}"
            )

        source_payload = read_json_url(
            QUERY_URL,
            {
                "where": "1=1",
                "outFields": "*",
                "returnGeometry": "true",
                "outSR": 4326,
                "f": "geojson",
            },
        )
        SOURCE_PATH.write_text(
            json.dumps(source_payload, ensure_ascii=False),
            encoding="utf-8",
        )
    except Exception as error:
        source_error = f"{type(error).__name__}: {error}"
        source_payload = {"type": "FeatureCollection", "features": []}
else:
    source_payload = json.loads(SOURCE_PATH.read_text(encoding="utf-8"))

print("Source mode:", source_mode)
print("Source records:", len(source_payload.get("features", [])))
if source_error:
    print("Source error:", source_error)

Source mode: cached dated snapshot
Source records: 153


## 5. Source quality checks and accepted value domains

The extraction stops for an unexpected schema or value domain because silently accepting a changed source would alter the meaning of the feature. Counts are reported rather than hard-coded: an annual update may legitimately change the number of establishments. The current snapshot should be reviewed for duplicate IDs, missing point geometry, WA coordinate bounds, open status, category, and Public/Private type.

In [4]:
required_source_fields = {
    "estab_id",
    "establishm",
    "is_open",
    "category",
    "est",
    "no_of_beds",
    "longitude",
    "latitude",
}

source_rows = []
geometry_errors = []
for position, feature in enumerate(source_payload.get("features", [])):
    properties = feature.get("properties") or {}
    geometry = feature.get("geometry") or {}
    coordinates = geometry.get("coordinates")
    if geometry.get("type") != "Point" or not coordinates or len(coordinates) < 2:
        geometry_errors.append(position)
        geometry_longitude, geometry_latitude = np.nan, np.nan
    else:
        geometry_longitude, geometry_latitude = coordinates[:2]
    row = dict(properties)
    row["_geometry_longitude"] = geometry_longitude
    row["_geometry_latitude"] = geometry_latitude
    source_rows.append(row)

health_source = pd.DataFrame(source_rows)

if source_error is None:
    missing_fields = required_source_fields - set(health_source.columns)
    if missing_fields:
        raise ValueError(f"Health source is missing fields: {sorted(missing_fields)}")
    if geometry_errors:
        raise ValueError(f"Source contains {len(geometry_errors)} invalid point geometries.")

    for column in ["is_open", "category", "est", "establishm"]:
        health_source[column] = health_source[column].astype("string").str.strip()

    for column in [
        "longitude",
        "latitude",
        "_geometry_longitude",
        "_geometry_latitude",
        "no_of_beds",
    ]:
        health_source[column] = pd.to_numeric(health_source[column], errors="coerce")

    def normalise_establishment_id(value):
        if pd.isna(value):
            return pd.NA
        number = float(value)
        return str(int(number)) if number.is_integer() else str(value).strip()

    health_source["estab_id"] = health_source["estab_id"].map(
        normalise_establishment_id
    ).astype("string")

    allowed_domains = {
        "is_open": {"Y", "N"},
        "category": {"Acute Hospital", "Nursing Post"},
        "est": {"Public", "Private"},
    }
    for column, allowed_values in allowed_domains.items():
        observed = set(health_source[column].dropna().unique())
        unexpected = observed - allowed_values
        if unexpected:
            raise ValueError(f"Unexpected {column} values: {sorted(unexpected)}")

    if health_source["estab_id"].isna().any():
        raise ValueError("The source contains missing establishment IDs.")
    if health_source["estab_id"].duplicated().any():
        duplicate_ids = health_source.loc[
            health_source["estab_id"].duplicated(keep=False), "estab_id"
        ].tolist()
        raise ValueError(f"Duplicate establishment IDs: {duplicate_ids}")

    valid_wa_geometry = (
        health_source["_geometry_latitude"].between(-36, -13)
        & health_source["_geometry_longitude"].between(112, 130)
    )
    if not valid_wa_geometry.all():
        raise ValueError("One or more source geometries fall outside broad WA bounds.")

    coordinate_difference = pd.concat(
        [
            (health_source["longitude"] - health_source["_geometry_longitude"]).abs(),
            (health_source["latitude"] - health_source["_geometry_latitude"]).abs(),
        ]
    ).max()

    print(f"Source records: {len(health_source):,}")
    print(f"Duplicate establishment IDs: {health_source['estab_id'].duplicated().sum():,}")
    print(f"Invalid point geometries: {len(geometry_errors):,}")
    print(f"Maximum attribute/geometry coordinate difference: {coordinate_difference:.6f}°")
    for column in ["is_open", "category", "est"]:
        print(f"\n{column} distribution:")
        print(health_source[column].value_counts(dropna=False).to_string())
    print("\nNumber Of Beds checks (excluded from features):")
    print(
        {
            "missing": int(health_source["no_of_beds"].isna().sum()),
            "zero": int(health_source["no_of_beds"].eq(0).sum()),
            "median": float(health_source["no_of_beds"].median()),
            "maximum": float(health_source["no_of_beds"].max()),
        }
    )

Source records: 153
Duplicate establishment IDs: 0
Invalid point geometries: 0
Maximum attribute/geometry coordinate difference: 0.000014°

is_open distribution:
is_open
Y    153

category distribution:
category
Acute Hospital    100
Nursing Post       53

est distribution:
est
Public     121
Private     32

Number Of Beds checks (excluded from features):
{'missing': 0, 'zero': 55, 'median': 11.0, 'maximum': 783.0}


## 6. Selected fields and modelling roles

| Output field | Meaning | Role in this 49-site project |
|---|---|---|
| `health_nearest_establishment_distance_m` | WGS84 ellipsoidal straight-line distance to the nearest `is_open == "Y"` record | The only first-baseline model candidate; compare models with and without it inside grouped/spatial CV |
| `health_nearest_establishment_category` | Official `Acute Hospital` or `Nursing Post` category | Diagnostic descriptor; do not automatically one-hot encode in this small sample |
| `health_nearest_establishment_type` | Official `Public` or `Private` type | Diagnostic descriptor; it does not imply that a person can access the service |
| `health_nearest_establishment_id` and `_name` | Official identifier and name | Audit and spot-check fields; never numeric or high-cardinality model inputs |
| `health_data_status` | `matched`, `no_open_record`, `invalid_coordinate`, or `source_failed` | Retrieval/QA status; not a behavioural predictor |

`no_of_beds` is rejected because zeros and facility categories are not yet comparable measures of current capacity. Radius counts are rejected because 153 statewide points are sparse and a chosen radius would be arbitrary. Network travel time, emergency capability, opening hours, and service quality are unavailable from this source and must not be inferred.

## 7. Calculate the nearest open establishment

The calculation uses the WGS84 ellipsoid through `pyproj.Geod`, not a degree-based Euclidean formula. This avoids UTM-zone management across statewide sites and is more accurate than a spherical approximation. It is still straight-line distance—not walking distance, driving distance, or travel time. Distance is rounded to 0.1 metre only for storage consistency; source-coordinate accuracy does not justify interpreting tiny differences.

In [5]:
feature_columns = [
    "site_id",
    "health_nearest_establishment_distance_m",
    "health_nearest_establishment_category",
    "health_nearest_establishment_type",
    "health_nearest_establishment_id",
    "health_nearest_establishment_name",
    "health_data_status",
]

if source_error is None:
    open_health = health_source.loc[health_source["is_open"].eq("Y")].copy()
else:
    open_health = pd.DataFrame()

geod = Geod(ellps="WGS84")
feature_rows = []

for site in sites.itertuples(index=False):
    row = {column: pd.NA for column in feature_columns}
    row["site_id"] = site.site_id

    if not site.valid_coordinate:
        row["health_data_status"] = "invalid_coordinate"
    elif source_error is not None:
        row["health_data_status"] = "source_failed"
    elif open_health.empty:
        row["health_data_status"] = "no_open_record"
    else:
        site_longitudes = np.full(len(open_health), float(site.longitude))
        site_latitudes = np.full(len(open_health), float(site.latitude))
        _, _, distances_m = geod.inv(
            site_longitudes,
            site_latitudes,
            open_health["_geometry_longitude"].to_numpy(),
            open_health["_geometry_latitude"].to_numpy(),
        )
        nearest_position = int(np.argmin(distances_m))
        nearest = open_health.iloc[nearest_position]

        row.update(
            {
                "health_nearest_establishment_distance_m": round(
                    float(distances_m[nearest_position]), 1
                ),
                "health_nearest_establishment_category": nearest["category"],
                "health_nearest_establishment_type": nearest["est"],
                "health_nearest_establishment_id": nearest["estab_id"],
                "health_nearest_establishment_name": nearest["establishm"],
                "health_data_status": "matched",
            }
        )

    feature_rows.append(row)

features = pd.DataFrame(feature_rows, columns=feature_columns)
features["site_id"] = features["site_id"].astype("string")
features["health_nearest_establishment_id"] = features[
    "health_nearest_establishment_id"
].astype("string")

if len(features) != len(sites):
    raise ValueError("Feature output row count no longer matches Model_Data.")
if features["site_id"].duplicated().any():
    raise ValueError("Feature output must contain exactly one row per site_id.")

print(features["health_data_status"].value_counts(dropna=False).to_string())
print(features.head().to_string(index=False))

health_data_status
matched    49
  site_id  health_nearest_establishment_distance_m health_nearest_establishment_category health_nearest_establishment_type health_nearest_establishment_id health_nearest_establishment_name health_data_status
250303-01                                  29476.9                        Acute Hospital                            Public                             412                   Harvey Hospital            matched
250303-02                                   2886.5                        Acute Hospital                            Public                             208                  Bunbury Hospital            matched
250303-03                                   1237.4                        Acute Hospital                            Public                             211                   Collie Hospital            matched
250303-04                                    485.9                        Acute Hospital                            Public             

## 8. Save the independent feature workbook

The workbook contains only `site_id` and source-prefixed health fields. Header styling, filters, frozen headers, and explicit distance formatting improve review without adding a second metadata sheet or copying `Model_Data`. Source definitions and copyright cautions remain visible in this notebook.

In [6]:
features.to_excel(
    FEATURE_PATH,
    sheet_name="WA_Health_Features",
    index=False,
)

workbook = load_workbook(FEATURE_PATH)
worksheet = workbook["WA_Health_Features"]
worksheet.freeze_panes = "A2"
worksheet.auto_filter.ref = worksheet.dimensions
worksheet.sheet_view.showGridLines = True

header_fill = PatternFill("solid", fgColor="1F4E78")
for cell in worksheet[1]:
    cell.fill = header_fill
    cell.font = Font(color="FFFFFF", bold=True)
    cell.alignment = Alignment(vertical="center", wrap_text=True)

widths = {
    "A": 15,
    "B": 45,
    "C": 43,
    "D": 39,
    "E": 38,
    "F": 43,
    "G": 23,
}
for column, width in widths.items():
    worksheet.column_dimensions[column].width = width
worksheet.row_dimensions[1].height = 42

for cell in worksheet["B"][1:]:
    cell.number_format = "0.0"

workbook.properties.title = "WA Health establishment features for model sites"
workbook.properties.subject = "Independent site_id feature table for CITS5553"
workbook.properties.creator = "CITS5553 Group 5"
workbook.save(FEATURE_PATH)

print("Saved:", FEATURE_PATH.relative_to(PROJECT_ROOT))

Saved: outputs/02_imagery_features/wa-health-establishments/haoqi-49-location-pilot/haoqi-04-wa-health-establishment-features.xlsx


## 9. Verify coverage, inspect contrasting sites, and test the team merge contract

The checks below report the actual feature distribution, review selected peri-urban/regional/remote contexts, and prove that the output merges one-to-one without changing the 49-row model foundation. The current sample contains no central Perth observations, so it cannot support a genuine metropolitan generalisation check.

In [7]:
matched = features.loc[features["health_data_status"].eq("matched")].copy()

print("Matched distance summary (metres):")
print(
    matched["health_nearest_establishment_distance_m"]
    .describe(percentiles=[0.25, 0.5, 0.75])
    .round(1)
    .to_string()
)
print("\nNearest category distribution:")
print(
    matched["health_nearest_establishment_category"]
    .value_counts(dropna=False)
    .to_string()
)
print("\nNearest type distribution:")
print(
    matched["health_nearest_establishment_type"]
    .value_counts(dropna=False)
    .to_string()
)

check_site_ids = ["250303-01", "250303-03", "250341-04", "260335-28"]
manual_check = (
    model_data[["site_id", "survey_location_name", "latitude", "longitude"]]
    .merge(features, on="site_id", how="left", validate="one_to_one")
    .loc[lambda frame: frame["site_id"].isin(check_site_ids)]
)
print("\nSelected manual checks:")
print(manual_check.to_string(index=False))

modeling_df = model_data.merge(
    features,
    on="site_id",
    how="left",
    validate="one_to_one",
)
training_df = modeling_df.loc[modeling_df["qa_status"].eq("ready")].copy()

if len(modeling_df) != len(model_data):
    raise ValueError("The feature merge changed the Model_Data row count.")
if modeling_df["health_data_status"].isna().any():
    raise ValueError("At least one model site did not receive a health data status.")

print(f"\nRows after one-to-one merge: {len(modeling_df):,}")
print(f"Training rows with qa_status == 'ready': {len(training_df):,}")
print("First-baseline model candidate: health_nearest_establishment_distance_m")

Matched distance summary (metres):
count       49.0
mean      4601.2
std       7323.3
min         94.5
25%        671.3
50%       1136.1
75%       3199.3
max      29476.9

Nearest category distribution:
health_nearest_establishment_category
Acute Hospital    44
Nursing Post       5

Nearest type distribution:
health_nearest_establishment_type
Public     45
Private     4

Selected manual checks:
  site_id           survey_location_name   latitude  longitude  health_nearest_establishment_distance_m health_nearest_establishment_category health_nearest_establishment_type health_nearest_establishment_id health_nearest_establishment_name health_data_status
250303-01 Melville Mandurah Hwy Mandurah -32.866282 115.705163                                  29476.9                        Acute Hospital                            Public                             412                   Harvey Hospital            matched
250303-03          Coalfields Hwy Collie -33.360624 116.154738                  